# 🎬 FHD ➜ 4K @ 60fps Video Upscaler

This notebook upscales a **1080p (Full HD)** video to **4K resolution at 60fps**, using two lightweight, fast, GPU-friendly open models pulled from **Hugging Face**:

| Task | Model | Why this one |
|---|---|---|
| Spatial upscaling (1080p → 4K, exact 2×) | **Real-ESRGAN (`RealESRGAN_x2plus`)** | The industry-standard fast GAN upscaler. Tiled inference + fp16 make it run comfortably on a free Colab T4 (16GB). Much faster than diffusion-based upscalers (SeedVR2, SUPIR, etc.) while still giving clean, non-plastic results on real footage. |
| Temporal upscaling (source fps → 60fps) | **RIFE (Real-Time Intermediate Flow Estimation, v4.x)** | The standard fast frame-interpolation network — real-time capable, no optical-flow pretraining needed, widely used in production pipelines. |

**Pipeline:** extract frames → upscale each frame (Real-ESRGAN, tiled/fp16) → interpolate frames up to 60fps (RIFE) → re-encode with ffmpeg → mux back the original audio.

**Interface:** the last cell launches a **Gradio** web app with `share=True`, which gives you a public link you can open on your phone or send to anyone — no local setup needed.

### How to use
1. Run every cell top-to-bottom (`Runtime ▸ Run all`).
2. Make sure `Runtime ▸ Change runtime type ▸ T4 GPU` is selected.
3. Open the public Gradio link printed by the last cell, upload your 1080p clip, and click **Upscale**.

> ⚠️ Free Colab GPUs (T4) have limited VRAM and session time. For your first run, test with a **short clip (5–15s)** before processing something long. The UI includes quality presets that trade speed/VRAM for quality — start with **Balanced**.


In [ ]:
#@title 1. Check GPU
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), "No GPU detected — go to Runtime ▸ Change runtime type ▸ T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1e9))


In [ ]:
#@title 2. Install dependencies
%%capture
!pip install -q basicsr facexlib realesrgan gradio huggingface_hub moviepy opencv-python-headless tqdm

# basicsr has a known incompatibility with newer torchvision (it imports a module
# that torchvision removed). Patch it so RealESRGAN keeps working on current Colab images.
#
# NOTE: we deliberately do NOT `import basicsr` here. basicsr's own __init__.py
# eagerly imports data/degradations.py, which is exactly the file with the broken
# torchvision import -- so importing basicsr before patching it would crash with
# the very error we're trying to fix. Instead we locate the file on disk directly.
import os, sys, site

basicsr_module_path = None
search_paths = list(site.getsitepackages())
try:
    search_paths.append(site.getusersitepackages())
except Exception:
    pass
search_paths.extend(sys.path)

for sp in search_paths:
    candidate = os.path.join(sp, "basicsr")
    if os.path.isdir(candidate):
        basicsr_module_path = candidate
        break

if basicsr_module_path is None:
    raise RuntimeError("Could not locate the basicsr installation directory.")

degr_file = os.path.join(basicsr_module_path, "data", "degradations.py")
if not os.path.exists(degr_file):
    raise FileNotFoundError(f"degradations.py not found at {degr_file}.")

with open(degr_file, "r") as f:
    content = f.read()
content = content.replace(
    "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
    "from torchvision.transforms.functional import rgb_to_grayscale",
)
with open(degr_file, "w") as f:
    f.write(content)
print("Patched basicsr for current torchvision. Done.")


In [ ]:
#@title 3. Download models + RIFE code from Hugging Face
import os, shutil
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs("/content/weights", exist_ok=True)

# --- Real-ESRGAN x2plus (spatial upscale 1080p -> 4K, exact 2x) ---
realesrgan_path = hf_hub_download(
    repo_id="dtarnow/UPscaler",
    filename="RealESRGAN_x2plus.pth",
    local_dir="/content/weights",
)
print("Real-ESRGAN weights:", realesrgan_path)

# --- RIFE architecture code (the RIFE_HDv3 package) ---
# We pull the whole rife/ folder from the official CogVideoX-5B Space. This gives a
# self-consistent set of files (RIFE_HDv3.py, IFNet_HDv3.py, warplayer.py, loss.py,
# refine.py, pytorch_msssim/, __init__.py) that are known to work together -- no need
# to clone Practical-RIFE (whose architecture .py files ship inside Google-Drive zips,
# not in the git repo, which is what broke the earlier version of this notebook).
snapshot_download(
    repo_id="zai-org/CogVideoX-5B-Space",
    repo_type="space",
    allow_patterns="rife/*",
    local_dir="/content/rife_pkg",
)
assert os.path.isfile("/content/rife_pkg/rife/RIFE_HDv3.py"), "RIFE_HDv3.py missing"
assert os.path.isfile("/content/rife_pkg/rife/IFNet_HDv3.py"), "IFNet_HDv3.py missing"
print("RIFE code ready in /content/rife_pkg/rife/")

# --- RIFE flownet weights (HDv3) ---
# This exact flownet.pkl (the classic ~12MB RIFE HDv3 checkpoint) is the one loaded into
# IFNet_HDv3 by ControlVideo, Sonic, LTX-Video and CogVideoX alike -- i.e. it matches the
# code we just downloaded. We drop it into a folder named so that Model.load_model() finds
# it as '<dir>/flownet.pkl'.
flownet_src = hf_hub_download(
    repo_id="jbilcke-hf/varnish",
    filename="rife/flownet.pkl",
    local_dir="/content/weights",
)
os.makedirs("/content/rife_train_log", exist_ok=True)
shutil.copy(flownet_src, "/content/rife_train_log/flownet.pkl")
print("RIFE weights ready at /content/rife_train_log/flownet.pkl")


In [ ]:
#@title 4. Imports & setup
import sys, os, cv2, gc, glob, shutil, subprocess, math, time
import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, "/content/rife_pkg")  # so `import rife.RIFE_HDv3` resolves

from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)
print("Using device:", device)


In [ ]:
#@title 5. Real-ESRGAN wrapper (tiled, fp16, VRAM-safe)

class FrameUpscaler:
    # Wraps RealESRGANer with tiling so it fits on a free-tier T4 (16GB).

    def __init__(self, model_path, tile=200, half=True):
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                         num_block=23, num_grow_ch=32, scale=2)
        self.upsampler = RealESRGANer(
            scale=2,
            model_path=model_path,
            model=model,
            tile=tile,          # smaller tile = less VRAM, a bit slower
            tile_pad=10,
            pre_pad=0,
            half=half,          # fp16 -> ~2x faster + less VRAM on T4
            device=device,
        )

    def upscale_bgr(self, img_bgr):
        # img_bgr: HxWx3 uint8 (OpenCV BGR). Returns upscaled HxWx3 uint8.
        output, _ = self.upsampler.enhance(img_bgr, outscale=2)
        return output

print("FrameUpscaler class ready.")


In [ ]:
#@title 6. RIFE wrapper (frame interpolation to 60fps)

from rife.RIFE_HDv3 import Model as RIFEModel  # code + weights downloaded in cell 3 (matched set)

class FrameInterpolator:
    def __init__(self, weights="/content/rife_train_log/flownet.pkl"):
        self.model = RIFEModel()
        # NOTE: we deliberately don't call self.model.load_model(). Its built-in loader
        # keeps only checkpoint keys containing 'module.' -- if this flownet.pkl was saved
        # WITHOUT that DataParallel prefix, that filter yields an empty dict and the load
        # silently fails. Loading here with a prefix-agnostic strip works either way.
        sd = torch.load(weights, map_location="cpu")
        sd = {k.replace("module.", ""): v for k, v in sd.items()}
        self.model.flownet.load_state_dict(sd, strict=True)
        self.model.eval()
        self.model.device()

    @staticmethod
    def _to_tensor(img_bgr):
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        t = torch.from_numpy(img).to(device, non_blocking=True).permute(2, 0, 1).float() / 255.0
        return t.unsqueeze(0)

    @staticmethod
    def _to_numpy(t):
        img = (t[0] * 255.0).clamp(0, 255).byte().permute(1, 2, 0).cpu().numpy()
        return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    @staticmethod
    def _pad(t, mult=32):
        _, _, h, w = t.shape
        ph = ((h - 1) // mult + 1) * mult
        pw = ((w - 1) // mult + 1) * mult
        return F.pad(t, (0, pw - w, 0, ph - h), mode="reflect"), h, w

    def middle_frame(self, img0_bgr, img1_bgr, scale=1.0):
        # Returns the temporal midpoint frame between two BGR frames.
        t0 = self._to_tensor(img0_bgr)
        t1 = self._to_tensor(img1_bgr)
        t0p, h, w = self._pad(t0)
        t1p, _, _ = self._pad(t1)
        with torch.no_grad():
            mid = self.model.inference(t0p, t1p, scale=scale)
        mid = mid[:, :, :h, :w]
        return self._to_numpy(mid)

print("FrameInterpolator class ready.")


In [ ]:
#@title 7. ffprobe helpers

def probe_video(path):
    def _run(args):
        return subprocess.run(args, capture_output=True, text=True).stdout.strip()

    fps_raw = _run(["ffprobe", "-v", "0", "-select_streams", "v:0",
                     "-show_entries", "stream=r_frame_rate",
                     "-of", "csv=p=0", path])
    num, den = fps_raw.split("/")
    fps = float(num) / float(den)

    nframes_raw = _run(["ffprobe", "-v", "0", "-select_streams", "v:0",
                         "-count_frames", "-show_entries", "stream=nb_read_frames",
                         "-of", "csv=p=0", path])
    nframes = int(nframes_raw) if nframes_raw.isdigit() else None

    has_audio = _run(["ffprobe", "-v", "0", "-select_streams", "a",
                       "-show_entries", "stream=index", "-of", "csv=p=0", path]) != ""

    w_raw = _run(["ffprobe", "-v", "0", "-select_streams", "v:0",
                  "-show_entries", "stream=width,height", "-of", "csv=p=0", path])
    width, height = (int(x) for x in w_raw.split(","))

    return {"fps": fps, "nframes": nframes, "has_audio": has_audio,
            "width": width, "height": height}

print("probe_video() ready.")


In [ ]:
#@title 8. Full pipeline: 1080p video -> 4K (optionally 60fps)

QUALITY_PRESETS = {
    # tile=0 means "no tiling" -- process the whole 1080p frame at once.
    # On a T4 with fp16, a full 1080p frame uses ~4-5 GB VRAM (out of 16 GB).
    "Max Speed (tile=0, uses ~10 GB)":   {"tile": 0,   "rife_scale": 1.0},
    "Balanced (tile=512, uses ~5 GB)":   {"tile": 512, "rife_scale": 1.0},
    "Safe (tile=192, uses ~3 GB)":       {"tile": 192, "rife_scale": 0.5},
}

# How many 4K frames to hold in RAM at once during RIFE interpolation.
# 100 frames × 3840×2160×3 bytes ≈ 2.4 GB — safe for Colab's ~12.7 GB RAM.
RIFE_CHUNK_SIZE = 100

def run_pipeline(input_path, preset="Max Speed (tile=0, uses ~10 GB)", target_fps=60,
                  max_seconds=None, interpolate=True, progress_cb=None):
    """
    Streaming pipeline -- never loads all frames into RAM at once.

    4K-only mode:  decode 1 frame → upscale → write to encoder.  Peak RAM: ~30 MB.
    4K+60fps mode: decode + upscale in chunks of RIFE_CHUNK_SIZE, interpolate
                   the chunk, flush to encoder, free, repeat.  Peak RAM: ~5 GB.
    """
    cfg = QUALITY_PRESETS[preset]

    def report(frac, desc):
        if progress_cb:
            progress_cb(frac, desc)
        print(f"[{frac*100:5.1f}%] {desc}")

    info = probe_video(input_path)
    src_fps = info["fps"]
    src_w, src_h = info["width"], info["height"]
    out_w, out_h = src_w * 2, src_h * 2
    frame_size_in = src_w * src_h * 3
    report(0.02, f"Source: {src_w}x{src_h} @ {src_fps:.2f}fps, "
                  f"audio={'yes' if info['has_audio'] else 'no'}")

    # ---- set up ffmpeg decoder (pipe out raw frames) ----------------------
    trim_args = ["-t", str(max_seconds)] if max_seconds else []
    decoder = subprocess.Popen(
        ["ffmpeg", "-hide_banner", "-loglevel", "error",
         "-i", input_path, *trim_args,
         "-f", "rawvideo", "-pix_fmt", "bgr24", "pipe:1"],
        stdout=subprocess.PIPE, bufsize=frame_size_in * 4,
    )

    # ---- set up ffmpeg encoder (pipe in raw frames) -----------------------
    if interpolate:
        exp = 0
        while src_fps * (2 ** exp) < target_fps:
            exp += 1
        exp = max(exp, 1)
        intermediate_fps = src_fps * (2 ** exp)
        final_fps = target_fps
    else:
        intermediate_fps = src_fps
        final_fps = src_fps

    silent_path = "/content/work/silent_4k.mp4"
    os.makedirs("/content/work", exist_ok=True)
    encoder = subprocess.Popen(
        ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
         "-f", "rawvideo", "-pix_fmt", "bgr24",
         "-s", f"{out_w}x{out_h}", "-r", str(intermediate_fps),
         "-i", "pipe:0",
         "-r", str(final_fps),
         "-c:v", "libx264", "-preset", "fast", "-crf", "18",
         "-pix_fmt", "yuv420p", silent_path],
        stdin=subprocess.PIPE, bufsize=out_w * out_h * 3 * 4,
    )

    # ---- helper: read one frame from decoder pipe -------------------------
    def read_frame():
        raw = decoder.stdout.read(frame_size_in)
        if len(raw) < frame_size_in:
            return None
        return np.frombuffer(raw, dtype=np.uint8).reshape(src_h, src_w, 3).copy()

    # ---- helper: write one 4K frame to encoder pipe -----------------------
    def write_frame(frame_bgr):
        encoder.stdin.write(frame_bgr.tobytes())

    # ---- load upscaler once -----------------------------------------------
    tile_desc = f"tile={cfg['tile']}" if cfg["tile"] > 0 else "no tiling (full frame)"
    report(0.05, f"Loading Real-ESRGAN ({tile_desc}, fp16)...")
    upscaler = FrameUpscaler("/content/weights/RealESRGAN_x2plus.pth",
                              tile=cfg["tile"], half=True)

    # ======================================================================
    # MODE A: 4K only (no RIFE) — pure streaming, one frame at a time
    # Peak RAM: just 1 source frame + 1 upscaled frame ≈ 30 MB
    # ======================================================================
    if not interpolate:
        report(0.08, "Streaming: decode → upscale → encode (one frame at a time)...")
        t_start = time.time()
        i = 0
        while True:
            frame = read_frame()
            if frame is None:
                break
            up = upscaler.upscale_bgr(frame)
            write_frame(up)
            i += 1
            if i % 10 == 0:
                elapsed = time.time() - t_start
                fps_rate = i / max(elapsed, 0.01)
                report(0.08 + 0.82 * min(i / max(info["nframes"] or 3600, 1), 1.0),
                       f"Frame {i} ({fps_rate:.1f} fr/s, "
                       f"elapsed {elapsed:.0f}s)")
        n = i
        report(0.90, f"Upscaled {n} frames in {time.time()-t_start:.0f}s "
                      f"({n/max(time.time()-t_start,0.01):.1f} fr/s)")

    # ======================================================================
    # MODE B: 4K + RIFE — chunked: upscale a batch, interpolate, flush
    # Each chunk ≈ RIFE_CHUNK_SIZE source frames.  After interpolation the
    # chunk may double in size (still fits in RAM).  Chunks overlap by 1
    # frame so RIFE can interpolate across chunk boundaries.
    # ======================================================================
    else:
        report(0.05, f"Loading RIFE for {src_fps:.0f} → {intermediate_fps:.0f} fps "
                      f"({exp} pass{'es' if exp > 1 else ''})...")
        rife = FrameInterpolator()

        t_start = time.time()
        total_written = 0
        chunk_idx = 0
        carry_frame = None  # last frame of previous chunk (for boundary interp)

        while True:
            # -- read & upscale a chunk --
            chunk = []
            if carry_frame is not None:
                chunk.append(carry_frame)
            for _ in range(RIFE_CHUNK_SIZE):
                frame = read_frame()
                if frame is None:
                    break
                up = upscaler.upscale_bgr(frame)
                chunk.append(up)

            if len(chunk) < 2:
                # fewer than 2 frames left — write any remaining and break
                for f in chunk:
                    if carry_frame is None or not np.array_equal(f, carry_frame):
                        write_frame(f)
                        total_written += 1
                break

            # -- interpolate the chunk --
            for level in range(exp):
                new_chunk = [chunk[0]]
                for j in range(len(chunk) - 1):
                    mid = rife.middle_frame(chunk[j], chunk[j + 1],
                                             scale=cfg["rife_scale"])
                    new_chunk.append(mid)
                    new_chunk.append(chunk[j + 1])
                chunk = new_chunk

            # -- flush chunk to encoder (skip the last frame — it becomes
            #    the carry for the next chunk to avoid a duplicate) --
            carry_frame = chunk[-1]
            for f in chunk[:-1]:
                write_frame(f)
                total_written += 1

            chunk_idx += 1
            elapsed = time.time() - t_start
            report(0.08 + 0.82 * min(chunk_idx * RIFE_CHUNK_SIZE /
                   max(info["nframes"] or 3600, 1), 1.0),
                   f"Chunk {chunk_idx} done, {total_written} frames written "
                   f"({elapsed:.0f}s elapsed)")
            del chunk
            gc.collect()

        # write final carry frame
        if carry_frame is not None:
            write_frame(carry_frame)
            total_written += 1

        del rife
        gc.collect()
        torch.cuda.empty_cache()
        report(0.90, f"Done: {total_written} frames in {time.time()-t_start:.0f}s")

    # ---- clean up pipes ---------------------------------------------------
    del upscaler
    gc.collect()
    torch.cuda.empty_cache()

    encoder.stdin.close()
    encoder.wait()
    decoder.stdout.close()
    decoder.wait()

    # ---- mux audio --------------------------------------------------------
    output_path = "/content/output_4k.mp4"
    report(0.95, "Muxing audio...")
    if info["has_audio"]:
        trim_args_a = ["-t", str(max_seconds)] if max_seconds else []
        subprocess.run([
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-i", silent_path,
            "-i", input_path, *trim_args_a,
            "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "copy", "-c:a", "aac", "-shortest",
            output_path,
        ], capture_output=True, text=True, check=True)
    else:
        shutil.copy(silent_path, output_path)

    report(1.0, "Done!")
    return output_path

print("run_pipeline() ready.")


In [ ]:
#@title 9. Gradio app (public shareable link)
import gradio as gr

def gradio_run(video_file, preset, interpolate, target_fps, limit_duration, max_seconds,
                progress=gr.Progress()):
    if video_file is None:
        raise gr.Error("Please upload a video first.")

    def cb(frac, desc):
        progress(frac, desc=desc)

    cap_seconds = float(max_seconds) if limit_duration else None
    out_path = run_pipeline(
        video_file,
        preset=preset,
        target_fps=int(target_fps),
        max_seconds=cap_seconds,
        interpolate=interpolate,
        progress_cb=cb,
    )
    return out_path, out_path

with gr.Blocks(title="FHD -> 4K Upscaler") as demo:
    gr.Markdown(
        "## 🎬 FHD ➜ 4K Video Upscaler\n"
        "Real-ESRGAN (spatial 2×) + optional RIFE (frame interpolation to 60fps).\n\n"
        "**For fastest results:** pick *Max Speed*, uncheck *Interpolate to 60fps*, "
        "and process your full clip."
    )
    with gr.Row():
        with gr.Column():
            video_in = gr.Video(label="Upload 1080p video", sources=["upload"])
            preset = gr.Radio(
                list(QUALITY_PRESETS.keys()),
                value="Max Speed (tile=0, uses ~10 GB)",
                label="Speed / VRAM preset",
            )
            interpolate = gr.Checkbox(
                value=False,
                label="Interpolate to 60fps (RIFE) — slower, uses more RAM",
            )
            target_fps = gr.Slider(24, 60, value=60, step=1,
                                    label="Target FPS (only if interpolation is on)")
            limit_duration = gr.Checkbox(
                value=False,
                label="Limit to first N seconds",
            )
            max_seconds = gr.Slider(1, 300, value=120, step=1,
                                     label="N seconds (only used if limit is checked)")
            run_btn = gr.Button("🚀 Upscale", variant="primary")
        with gr.Column():
            video_out = gr.Video(label="Result (4K)")
            file_out = gr.File(label="Download file")

    run_btn.click(
        gradio_run,
        inputs=[video_in, preset, interpolate, target_fps, limit_duration, max_seconds],
        outputs=[video_out, file_out],
    )

demo.queue().launch(share=True, debug=True)


### Notes & troubleshooting

- **Streaming pipeline**: frames are decoded, upscaled, and encoded **one at a time** (4K-only mode) or in small chunks (RIFE mode). Peak system RAM usage is ~30 MB for 4K-only, ~2.5 GB for RIFE — not the 22+ GB that crashed before.
- **Speed presets:**
  - **Max Speed (tile=0)**: entire 1080p frame processed at once — uses ~8-10 GB VRAM, fastest throughput.
  - **Balanced (tile=512)**: moderate tiling — ~5 GB VRAM.
  - **Safe (tile=192)**: heavy tiling — ~3 GB VRAM, slowest.
- **4K-only (no RIFE)**: uncheck "Interpolate to 60fps" — keeps your original frame rate, skips RIFE entirely, and runs in pure streaming mode with almost zero RAM overhead.
- **RIFE chunked mode**: when interpolation is enabled, the pipeline processes 100 upscaled frames at a time, interpolates the chunk, writes to the encoder, and frees the memory before the next chunk. Chunks overlap by 1 frame for seamless boundaries.
- **2-minute 1080p clip estimates (Max Speed preset on free T4):**
  - 4K only: ~10-15 minutes
  - 4K + 60fps: ~25-40 minutes
- **"CUDA out of memory"** → switch from Max Speed to Balanced.
- **Audio** is copied from the original and muxed back untouched.
